In [ ]:
import os
import random
import shutil
import yaml


"""
    1. Shuffle the images in `raw/images`.
    2. Split them into `train`, `valid`, and `test` directories based on the specified ratios.
    3. Copy the images and their corresponding labels (if they exist) to the respective directories.
    4. Update the `data.yaml` file to reflect the new splits.
    5. Create a new `data_split.yaml` file with the updated paths
    6. Print the number of images in each split.  

"""

random.seed(42)

dataset_root = "dataset/GeneST_final_data"

raw_img_dir = os.path.join(dataset_root, "raw", "images")
raw_lbl_dir = os.path.join(dataset_root, "raw", "labels")
splits = ["train", "valid", "test"]
ratios = [0.90, 0.05, 0.05]

for split in splits:
    os.makedirs(os.path.join(dataset_root, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, split, "labels"), exist_ok=True)

all_imgs = [
    f for f in os.listdir(raw_img_dir)
    if f.lower().endswith(".jpg")
]
random.shuffle(all_imgs)

N = len(all_imgs)
n_train = int(ratios[0] * N)
n_valid = int(ratios[1] * N)

split_assignments = {
    "train": all_imgs[:n_train],
    "valid": all_imgs[n_train:n_train + n_valid],
    "test": all_imgs[n_train + n_valid:]
}

for split, img_list in split_assignments.items():
    for img_name in img_list:
        src_img = os.path.join(raw_img_dir, img_name)
        dst_img = os.path.join(dataset_root, split, "images", img_name)
        shutil.copy2(src_img, dst_img)
        base = os.path.splitext(img_name)[0]
        lbl_name = base + ".txt"
        src_lbl = os.path.join(raw_lbl_dir, lbl_name)
        dst_lbl = os.path.join(dataset_root, split, "labels", lbl_name)

        if os.path.exists(src_lbl):
            shutil.copy2(src_lbl, dst_lbl)
        else:
            print(f"Missing label for {img_name}")

with open(os.path.join(dataset_root, "data.yaml")) as f:
    data = yaml.safe_load(f)
data["train"] = "train/images"
data["val"  ] = "valid/images"
data["test" ] = "test/images" 

with open(os.path.join(dataset_root, "data_split.yaml"), "w") as f:
    yaml.dump(data, f)

print(f"Done! {N} images split into "
      f"{len(split_assignments['train'])} train, "
      f"{len(split_assignments['valid'])} valid, "
      f"{len(split_assignments['test'])} test.")


In [2]:
# !nvidia-smi

In [ ]:
from ultralytics import YOLO
from ultralytics.models.yolo.segment import SegmentationTrainer
from ultralytics.utils.loss import FocalLoss
from ultralytics.engine.trainer import BaseTrainer
# from ultralytics.models.yolo.segment.train import SegTrainer
import glob
import os, glob, cv2, matplotlib.pyplot as plt

In [3]:
import os
DATASET_YAML = "dataset/GeneST_final_data/data.yaml"
assert os.path.isfile(DATASET_YAML), f"{DATASET_YAML} not found"

In [4]:
# class FocalLossTrainer(BaseTrainer):
#     def get_criterion(self):
#         criterion = super().get_criterion()
#         # Replace the classification loss component with FocalLoss
#         # Keep gamma and alpha as parameters you can tune
#         criterion.bce = FocalLoss()
#         return criterion

In [5]:
model = YOLO("yolov8x-seg.pt")  

In [ ]:
"""
    1. Load the YOLOv8x segmentation model.
    2. Set the training parameters such as task, data, image size, epochs,
       batch size, number of workers, device, optimizer, learning rate,
       learning rate final, weight decay, warmup epochs, patience,
       mixed precision (amp), caching, seed, mosaic augmentation,
       mixup augmentation, copy-paste augmentation, HSV augmentation,
       geometric transformations (degrees, translate, scale, shear, perspective),
       horizontal and vertical flipping.
    3. Specify the project and name for saving the training results.         
    
"""


model.train(
    
    task="segment",
    data=DATASET_YAML,
    imgsz=1024,
    epochs=200,              
    batch=32,
    workers=4,
    device="0,1,2,3",               
    optimizer="AdamW",
    lr0=5e-4,                 
    lrf=0.05, 
    weight_decay=0.05,
    warmup_epochs=3,
    patience=100,             
    amp=True,
    cache=True,
    seed=0,
    mosaic=0.0,              
    # close_mosaic=10,         
    mixup=0.0,               
    copy_paste=0.1,          
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.10,   
    degrees=15.0,
    translate=0.05,
    scale=0.10,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.5,

    project="crypt_segment_3",
    name="yolov8x_seg",
)

In [ ]:
RUN_ROOT       = "crypt_segment_3/yolov8x_seg"      
BEST_WEIGHTS   = f"{RUN_ROOT}/weights/best.pt"
LAST_WEIGHTS   = f"{RUN_ROOT}/weights/last.pt"
DATASET_YAML   = "dataset/GeneST_final_data/data.yaml"      

assert os.path.isfile(BEST_WEIGHTS) and os.path.isfile(LAST_WEIGHTS), "weights not found!"
assert os.path.isfile(DATASET_YAML), "dataset.yaml missing!"

In [ ]:
test_dir  = os.path.join(os.path.dirname(DATASET_YAML), "test", "images")
test_imgs = sorted(glob.glob(os.path.join(test_dir, "*.*")))
print(f"🖼️  {len(test_imgs)} test images found in {test_dir}")

Lower it (e.g. 0.05) → to see more detections, including weak ones.
Raise it (e.g. 0.50) → to see fewer, but highly confident, detections.

In [ ]:
def run_predict(weights_path, subdir):

    """ Run inference on the test images using the specified weights. 
    Args:
        weights_path (str): Path to the model weights file.
        subdir (str): Subdirectory name for saving results.

    Returns:
        list: List of results from the prediction. 
    Note:
        - `source` should be a list of paths to the images you want to predict on
        - `imgsz` is the size of the images to be used for inference.
        - `conf` is the confidence threshold for predictions.
        - `save` indicates whether to save the predictions as images.
        - `save_txt` indicates whether to save the predictions in text format.
        - `save_conf` indicates whether to save the confidence scores in the text files.
        - `project` is the root directory where results will be saved.
        - `name` is the name of the subdirectory where results will be saved.
        - `exist_ok` allows overwriting the existing results directory if it exists.
    
    """

    model = YOLO(weights_path)
    return model.predict(
        source=test_imgs,          
        imgsz=1024,
        conf=0.50,
        save=True,                
        save_txt=True,            
        save_conf=False,
        project=RUN_ROOT,          
        name=subdir,              
        exist_ok=True              
    )

In [ ]:
print("Inference with best.pt …")
best_results = run_predict(BEST_WEIGHTS, "results_best_B1")

In [ ]:
print("Inference with last.pt …")
_ = run_predict(LAST_WEIGHTS, "results_last_B1")

In [ ]:

fig, ax = plt.subplots(2, 3, figsize=(18, 10))
for a, res in zip(ax.ravel(), best_results[:6]):     
    overlay_path = os.path.join(res.save_dir, os.path.basename(res.path))
    a.imshow(cv2.cvtColor(cv2.imread(overlay_path), cv2.COLOR_BGR2RGB))
    a.set_title(os.path.basename(res.path))
    a.axis("off")
plt.tight_layout(); plt.show()